# YouTube Speaker Diarization

This notebook downloads a YouTube video, transcribes it with speaker labels using pyannote.audio, and outputs a formatted transcript.

**Setup:**
1. Runtime → Change runtime type → T4 GPU
2. Get a HuggingFace token from https://huggingface.co/settings/tokens
3. Accept pyannote model terms: https://huggingface.co/pyannote/speaker-diarization-3.1
4. (Optional) Add GitHub token for error reporting

In [ ]:
#@title 0. Setup GitHub Reporting (Optional)
#@markdown This allows the notebook to report errors/success to GitHub Issues
#@markdown so fixes can be pushed quickly.
#@markdown 
#@markdown Create a token at: https://github.com/settings/tokens
#@markdown (needs 'repo' scope for private repos, or 'public_repo' for public)

GITHUB_TOKEN = ""  #@param {type:"string"}
GITHUB_REPO = "altbier/claudecode"  #@param {type:"string"}
ENABLE_REPORTING = True  #@param {type:"boolean"}

import json
import urllib.request
import urllib.error
import traceback
import sys
from datetime import datetime

class GitHubReporter:
    def __init__(self, token, repo, enabled=True):
        self.token = token
        self.repo = repo
        self.enabled = enabled and bool(token)
        self.logs = []
        self.step = 0
        
    def log(self, message):
        """Add to internal log."""
        timestamp = datetime.now().strftime("%H:%M:%S")
        entry = f"[{timestamp}] {message}"
        self.logs.append(entry)
        print(entry)
    
    def start_step(self, name):
        """Mark the start of a step."""
        self.step += 1
        self.log(f"Step {self.step}: {name}")
    
    def _get_auth_header(self):
        """Return correct auth header based on token type."""
        if self.token.startswith("github_pat_"):
            return f"Bearer {self.token}"
        return f"token {self.token}"
    
    def _post_issue(self, title, body, labels=None):
        """Post an issue to GitHub."""
        if not self.enabled:
            print("GitHub reporting disabled (no token)")
            return None
            
        url = f"https://api.github.com/repos/{self.repo}/issues"
        data = {"title": title, "body": body}
        if labels:
            data["labels"] = labels
        
        req = urllib.request.Request(
            url,
            data=json.dumps(data).encode('utf-8'),
            headers={
                "Authorization": self._get_auth_header(),
                "Accept": "application/vnd.github.v3+json",
                "Content-Type": "application/json",
            },
            method="POST"
        )
        
        try:
            with urllib.request.urlopen(req) as response:
                result = json.loads(response.read().decode())
                print(f"Posted to GitHub: {result['html_url']}")
                return result['html_url']
        except urllib.error.HTTPError as e:
            body_text = e.read().decode() if e.fp else ""
            print(f"Failed to post to GitHub: {e.code} {e.reason}")
            print(f"Response: {body_text[:200]}")
            if labels and e.code == 422:
                print("Retrying without labels...")
                return self._post_issue(title, body, labels=None)
            return None
    
    def report_error(self, error, cell_name="Unknown"):
        """Report an error to GitHub."""
        tb = traceback.format_exc()
        
        gpu_info = "Unknown"
        try:
            import torch
            gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"
        except:
            pass
        
        config_lines = []
        for var in ['VIDEO_ID', 'START_MIN', 'END_MIN']:
            val = globals().get(var, 'not set')
            config_lines.append(f"- {var}: `{val}`")
        
        body = f"""## Colab Notebook Error

**Cell:** {cell_name}
**Time:** {datetime.now().isoformat()}
**GPU:** {gpu_info}

### Error
```
{type(error).__name__}: {str(error)}
```

### Traceback
```python
{tb}
```

### Execution Log
```
{chr(10).join(self.logs[-20:])}
```

### Configuration
{chr(10).join(config_lines)}
"""
        
        title = f"[Colab Error] {cell_name}: {type(error).__name__}"
        return self._post_issue(title, body, labels=["colab", "bug"])
    
    def report_success(self, summary=""):
        """Report successful completion."""
        body = f"""## Colab Notebook Success

**Time:** {datetime.now().isoformat()}

### Summary
{summary}

### Execution Log
```
{chr(10).join(self.logs)}
```
"""
        
        title = f"[Colab Success] Completed at {datetime.now().strftime('%Y-%m-%d %H:%M')}"
        return self._post_issue(title, body, labels=["colab", "success"])


# Initialize reporter
reporter = GitHubReporter(GITHUB_TOKEN, GITHUB_REPO, ENABLE_REPORTING)
reporter.log("GitHub reporter initialized")

if GITHUB_TOKEN:
    print(f"Reporting enabled for: {GITHUB_REPO}")
else:
    print("No GitHub token - reporting disabled (errors will only show in Colab)")

In [ ]:
#@title 1. Install Dependencies (restarts runtime on first run)
#@markdown After first install, the runtime will restart automatically.
#@markdown This is required because pyannote upgrades numpy.
#@markdown **Just re-run all cells after restart — it will skip this step.**

import os

if not os.path.exists('/tmp/.deps_installed'):
    reporter.start_step("Install Dependencies")
    !pip install -q "pyannote.audio>=3.1,<4" "yt-dlp>=2024" "youtube-transcript-api>=0.6"
    # Install deno (yt-dlp's default JS runtime for YouTube extraction)
    !curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1
    !ln -sf /root/.deno/bin/deno /usr/local/bin/deno
    reporter.log("Dependencies + deno installed — restarting runtime...")

    # Mark as done so we skip on re-run
    with open('/tmp/.deps_installed', 'w') as f:
        f.write('done')

    # Restart to pick up new numpy
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed, skipping.")

In [ ]:
#@title 2. Configuration
reporter.start_step("Configuration")

import re

YOUTUBE_URL = "https://www.youtube.com/watch?v=a0UCcZWx9ao"  #@param {type:"string"}
HF_TOKEN = ""  #@param {type:"string"}
START_MIN = 3  #@param {type:"integer"}
END_MIN = 15  #@param {type:"integer"}

START_SEC = START_MIN * 60
END_SEC = END_MIN * 60

def extract_video_id(url):
    match = re.search(r'(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})', url)
    return match.group(1) if match else url

VIDEO_ID = extract_video_id(YOUTUBE_URL)
reporter.log(f"Video ID: {VIDEO_ID}")
reporter.log(f"Time range: {START_MIN}:00 - {END_MIN}:00")

In [ ]:
#@title 3. Download Audio from YouTube
reporter.start_step("Download Audio")

try:
    import yt_dlp
    import os
    import glob

    # Clean up any previous downloads
    for f in glob.glob('audio*'):
        os.remove(f)

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
        }],
        'outtmpl': 'audio.%(ext)s',
        'quiet': False,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(YOUTUBE_URL, download=True)
        video_title = info.get('title', 'Unknown')

    # Find the wav file
    wav_files = glob.glob('audio*.wav')
    if not wav_files:
        raise FileNotFoundError("No audio file produced. yt-dlp may have failed to download.")
    audio_file = wav_files[0]
    reporter.log(f"Downloaded: {video_title}")
    reporter.log(f"Audio file: {audio_file}")

except Exception as e:
    reporter.report_error(e, "Download Audio")
    raise

In [ ]:
#@title 4. Trim Audio to Time Range
reporter.start_step("Trim Audio")

try:
    import subprocess

    trim_cmd = [
        'ffmpeg', '-y',
        '-i', audio_file,
        '-ss', str(START_SEC),
        '-to', str(END_SEC),
        '-c', 'copy',
        'audio_trimmed.wav',
        '-loglevel', 'error'
    ]

    subprocess.run(trim_cmd, check=True)
    reporter.log(f"Trimmed to {START_MIN}:00 - {END_MIN}:00")
    reporter.log(f"Duration: {END_MIN - START_MIN} minutes")

except Exception as e:
    reporter.report_error(e, "Trim Audio")
    raise

In [ ]:
#@title 5. Run Speaker Diarization (pyannote)
reporter.start_step("Speaker Diarization")

try:
    from pyannote.audio import Pipeline
    import torch

    if not HF_TOKEN:
        raise ValueError("Please set HF_TOKEN in the Configuration cell!")

    reporter.log("Loading pyannote pipeline...")
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        token=HF_TOKEN
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    reporter.log(f"Using device: {device}")
    pipeline.to(device)

    reporter.log("Running diarization...")
    diarization = pipeline("audio_trimmed.wav")

    speakers_found = set(label for _, _, label in diarization.itertracks(yield_label=True))
    reporter.log(f"Speakers found: {len(speakers_found)}")

except Exception as e:
    reporter.report_error(e, "Speaker Diarization")
    raise

In [ ]:
#@title 6. Fetch YouTube Transcript
reporter.start_step("Fetch Transcript")

try:
    from youtube_transcript_api import YouTubeTranscriptApi

    api = YouTubeTranscriptApi()
    transcript_data = api.fetch(VIDEO_ID)

    # Filter to time range and adjust timestamps to be relative to trimmed audio
    segments = []
    for s in transcript_data:
        seg_start = s.start
        seg_end = s.start + s.duration
        
        # Include if segment overlaps with our time range
        if seg_end >= START_SEC and seg_start <= END_SEC:
            segments.append({
                'text': s.text,
                'start': s.start - START_SEC,  # Relative to trimmed audio
                'end': seg_end - START_SEC,
                'start_absolute': s.start,  # Keep original for display
            })

    reporter.log(f"Transcript segments: {len(segments)}")

except Exception as e:
    reporter.report_error(e, "Fetch Transcript")
    raise

In [ ]:
#@title 7. Merge Diarization with Transcript
reporter.start_step("Merge Diarization")

try:
    def get_speaker_at_time(diarization, time_sec):
        """Find which speaker is talking at a given time."""
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            if turn.start <= time_sec <= turn.end:
                return speaker
        return None

    # Assign speakers to transcript segments
    for seg in segments:
        # Use midpoint of segment to determine speaker
        mid_time = (seg['start'] + seg['end']) / 2
        # Clamp to valid range
        mid_time = max(0, mid_time)
        seg['speaker'] = get_speaker_at_time(diarization, mid_time)

    # Get unique speakers in order of appearance
    speakers_seen = []
    for seg in segments:
        if seg['speaker'] and seg['speaker'] not in speakers_seen:
            speakers_seen.append(seg['speaker'])

    # Create friendly names
    speaker_names = {spk: f"Speaker {i+1}" for i, spk in enumerate(speakers_seen)}

    reporter.log(f"Unique speakers: {len(speaker_names)}")
    for orig, name in speaker_names.items():
        print(f"  {orig} → {name}")

except Exception as e:
    reporter.report_error(e, "Merge Diarization")
    raise

In [ ]:
#@title 8. Generate Formatted Transcript
reporter.start_step("Generate Output")

try:
    def format_time(seconds):
        """Format seconds as MM:SS"""
        total_sec = int(seconds)
        return f"{total_sec // 60:02d}:{total_sec % 60:02d}"

    output_lines = [
        f"# {video_title}",
        "",
        f"**Video:** {YOUTUBE_URL}",
        f"**Time Range:** {START_MIN}:00 - {END_MIN}:00",
        f"**Speakers Identified:** {len(speaker_names)}",
        "",
        "## Transcript",
        ""
    ]

    current_speaker = None
    for seg in segments:
        speaker = speaker_names.get(seg['speaker'], 'Unknown')
        timestamp = format_time(seg['start_absolute'])  # Use absolute time for display
        
        if speaker != current_speaker:
            current_speaker = speaker
            output_lines.append(f"\n**{speaker}** [{timestamp}]")
        
        # Clean up >> markers from YouTube transcript
        text = re.sub(r'^>>+\s*', '', seg['text'])
        output_lines.append(text)

    transcript_output = "\n".join(output_lines)

    # Save to file
    output_filename = f"transcript_{VIDEO_ID}_{START_MIN}-{END_MIN}min.md"
    with open(output_filename, 'w') as f:
        f.write(transcript_output)

    reporter.log(f"Saved to {output_filename}")
    print("\n" + "="*50)
    print(transcript_output[:3000])
    if len(transcript_output) > 3000:
        print("\n[... truncated ...]")

except Exception as e:
    reporter.report_error(e, "Generate Output")
    raise

In [ ]:
#@title 9. Report Success & Download
reporter.start_step("Complete")

summary = f"""
- **Video:** {video_title}
- **Time Range:** {START_MIN}:00 - {END_MIN}:00
- **Speakers Found:** {len(speaker_names)}
- **Transcript Segments:** {len(segments)}
- **Output File:** {output_filename}
"""

reporter.log("Pipeline complete!")

# Report success to GitHub
if ENABLE_REPORTING and GITHUB_TOKEN:
    reporter.report_success(summary)

# Download the file
from google.colab import files
files.download(output_filename)

In [ ]:
#@title 10. (Optional) View Diarization Timeline
print("Speaker Timeline:")
print("="*60)
for turn, _, speaker in diarization.itertracks(yield_label=True):
    start_abs = turn.start + START_SEC
    end_abs = turn.end + START_SEC
    name = speaker_names.get(speaker, speaker)
    print(f"{format_time(start_abs)} - {format_time(end_abs)}: {name}")